In [ ]:
import numpy as np
import pandas as pd

# Edit the path to your dataset
path = "../../../../assets/datasets/3_KG_Chronic_Kidney_Disease_(CKD)_Clinical_Dataset/data/Training_CKD_dataset.csv"

# Edit the name of the target column
target = "Target"

# Edit the null value symbol 
df = pd.read_csv(path, na_values=["?"])
df.head()

,Target,Age,Gender,BMI,Systolic_BP,Diastolic_BP,Heart_Rate,Serum_Creatinine,Blood_Urea_Nitrogen,eGFR,...,Fasting_Glucose,HbA1c,Cholesterol,Triglycerides,Serum_Albumin,Total_Protein,Diabetes,Hypertension,Smoking_Status,Family_History_Kidney
0,Healthy Kidney,29,1,28,97,69,99,0,12,95,...,96,7.547874,204,120,4,7.091259,Yes,Yes,Yes,Yes
1,Severe CKD (Stage 4),43,0,18,165,100,67,5,87,28,...,88,7.287338,166,277,2,7.875167,Yes,Yes,Yes,No
2,Healthy Kidney,77,0,32,116,63,101,0,16,100,...,82,9.114854,246,299,4,7.083558,No,No,Yes,No
3,Healthy Kidney,83,0,24,93,75,87,0,10,101,...,82,7.286450,173,285,4,6.428780,Yes,No,No,Yes
4,Healthy Kidney,38,1,19,111,70,92,0,10,102,...,106,8.376492,266,294,4,7.852894,Yes,No,Yes,No


In [3]:
def column_type(s):
    n = s.nunique()
    if pd.api.types.is_bool_dtype(s):
        return "boolean"
    if pd.api.types.is_numeric_dtype(s):
        if n == 2:
            return "numeric (binary)"
        if pd.api.types.is_integer_dtype(s):
            return "numeric (integer)"
        return "numeric (float)"
    if n <= 20 or n / max(len(s), 1) < 0.05:
        return "categorical (string)"
    return "string (free text / id)"


def has_classes(s, max_discrete=10):
    kind = column_type(s)
    if kind in ("boolean", "numeric (binary)") or kind.startswith("categorical"):
        return True
    return kind == "numeric (integer)" and s.nunique() <= max_discrete

In [4]:
rows, cols = df.shape
y = df[target]
valid_rows = len(df.dropna())

summary = pd.DataFrame({
    "type": pd.Series({c: column_type(df[c]) for c in df.columns}),
    "unique": df.nunique(),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
})

summary["classes"] = pd.Series({
    c: sorted(df[c].dropna().unique().tolist(), key=str) if has_classes(df[c]) else "-"
    for c in df.columns
})

In [5]:
print("DATASET OVERVIEW")
print(f"Instances (rows):            {rows}")
print(f"Columns:                     {cols}")
print(f"Features (excl. target):     {cols - 1}")
print(f"Valid instances (no NaN):    {valid_rows} ({valid_rows / rows:.2%})")
print(f"Instances with any NaN:      {rows - valid_rows}")
print(f"Duplicate rows:              {df.duplicated().sum()}")
print(f"Memory:                      {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"Features per instance ratio: {(cols - 1) / rows:.4f}")

DATASET OVERVIEW
Instances (rows):            21000
Columns:                     36
Features (excl. target):     35
Valid instances (no NaN):    21000 (100.00%)
Instances with any NaN:      0
Duplicate rows:              0
Memory:                      10784.0 KB
Features per instance ratio: 0.0017


In [6]:
print("COLUMN TYPES AND CLASSES")
print(summary.to_string())

print("\nTYPE COUNTS")
print(summary["type"].value_counts().to_string())

constant = summary.index[summary["unique"] <= 1].tolist()
print(f"\nConstant columns: {constant if constant else 'none'}")

COLUMN TYPES AND CLASSES
                                          type  unique  missing  missing_%                                                                                                         classes
Target                    categorical (string)       5        0        0.0  [Healthy Kidney, Kidney Failure (Stage 5), Mild CKD (Stage 1–2), Moderate CKD (Stage 3), Severe CKD (Stage 4)]
Age                          numeric (integer)      65        0        0.0                                                                                                               -
Gender                        numeric (binary)       2        0        0.0                                                                                                          [0, 1]
BMI                          numeric (integer)      17        0        0.0                                                                                                               -
Systolic_BP                  numeric (in

In [7]:
print(f"TARGET: {target}")
if has_classes(y) or not pd.api.types.is_numeric_dtype(y):
    counts = y.value_counts(dropna=False)
    share = (counts / len(y) * 100).round(2)
    print("Task:                  classification")
    print(f"Number of classes:     {y.nunique()}")
    print(f"Class values:          {sorted(y.dropna().unique().tolist(), key=str)}")
    print(pd.DataFrame({"count": counts, "%": share}).to_string())
    print(f"Imbalance ratio (max/min): {counts.max() / counts.min():.2f}")
    print(f"Majority-class baseline:   {counts.max() / len(y):.2%}")
else:
    print("Task:                  regression")
    print(y.describe().to_string())

TARGET: Target
Task:                  classification
Number of classes:     5
Class values:          ['Healthy Kidney', 'Kidney Failure (Stage 5)', 'Mild CKD (Stage 1–2)', 'Moderate CKD (Stage 3)', 'Severe CKD (Stage 4)']
                          count      %
Target                                
Healthy Kidney            15744  74.97
Mild CKD (Stage 1–2)       2491  11.86
Moderate CKD (Stage 3)     1489   7.09
Severe CKD (Stage 4)        856   4.08
Kidney Failure (Stage 5)    420   2.00
Imbalance ratio (max/min): 37.49
Majority-class baseline:   74.97%


In [8]:
numeric = df.select_dtypes(include=[np.number])
if not numeric.empty:
    print("NUMERIC FEATURE STATS")
    stats = numeric.describe().T[["min", "max", "mean", "std"]]
    stats["skew"] = numeric.skew()
    print(stats.round(3).to_string())

NUMERIC FEATURE STATS
                                 min        max        mean        std   skew
Age                           20.000      84.00      51.952     18.796 -0.000
Gender                         0.000       1.00       0.501      0.500 -0.002
BMI                           18.000      34.00      25.978      4.890  0.001
Systolic_BP                   90.000     189.00     113.498     19.152  1.320
Diastolic_BP                  60.000     119.00      75.281     12.107  1.131
Heart_Rate                    60.000     109.00      84.281     14.388  0.020
Serum_Creatinine               0.000       9.00       0.630      1.482  3.145
Blood_Urea_Nitrogen            7.000     149.00      21.682     20.800  3.033
eGFR                           5.000     119.00      91.426     26.787 -1.555
Urine_Albumin                  0.000     999.00      59.987    136.149  3.732
Urine_Protein                  0.000     499.00      31.221     73.192  3.683
Albumin_Creatinine_Ratio       0.000    11